# Model 1 — Custom CNN

A small convolutional network is the transparent baseline. It is quick to train, easy to modify in PyCharm, and gives the project a reproducible reference point before transfer learning.

In [1]:
from pathlib import Path
import json, math, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(p for p in candidates if (p / 'app').is_dir() and (p / 'requirements.txt').exists())
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_SIZE, SEED
from app.data import load_manifest, split_manifest
from app.labels import CLASS_NAMES
from app.metrics import calculate_classification_metrics
from app.model import build_model
tf.keras.utils.set_random_seed(SEED)
print('Service:', SERVICE_DIR)
print('Dataset:', DATASET_CSV)

I0000 00:00:1787654404.675044    1100 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787654404.741816    1100 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787654405.942265    1100 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Service: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service
Dataset: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/Dataset/FracAtlas/dataset.csv


In [2]:
frame = load_manifest()
train, validation, test = split_manifest(frame)
print(f'Usable images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

W0000 00:00:1787654439.924826    1100 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1787654439.930401    1100 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1787654440.182824    1100 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1787654477.919788    1100 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 414/454
W0000 00:00:1787654477.919913    1100 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop win

Usable images: 4,024 | train: 3,219 | validation: 402 | test: 403


/tmp/ipykernel_1100/2782770702.py:1: RuntimeWarning: Skipped 59 unreadable image file(s) from the manifest.
  frame = load_manifest()


,count
label,
NO_FRACTURE,3307
ONE_FRACTURE,546
MULTIPLE_FRACTURES,171


In [3]:
BATCH_SIZE = 32
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, IMAGE_SIZE), label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_dataset = make_dataset(train, True).repeat()
validation_dataset = make_dataset(validation).repeat()
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'])
class_weights = {i: float(value) for i, value in enumerate(weights)}
print('Class weights:', class_weights)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Class weights: {0: 0.4056710775047259, 1: 2.4553775743707096, 2: 7.8321167883211675}


In [4]:
model = build_model()
model.summary()
model_path = ARTIFACT_DIR / 'models' / 'custom_cnn.keras'
model_path.parent.mkdir(parents=True, exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
]
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=20, steps_per_epoch=TRAIN_STEPS, validation_steps=VALIDATION_STEPS, class_weight=class_weights, callbacks=callbacks, shuffle=False)

Model: "fracatlas_fracture_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xray (InputLayer)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 110,819 (432.89 KB)

 Trainable params: 110,371 (431.14 KB)

 Non-trainable params: 448 (1.75 KB)

...

I0000 00:00:1787654574.829198    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 5969345449245374614
I0000 00:00:1787654574.829262    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 18332283301707709952
I0000 00:00:1787654574.829270    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 5255449077192355526


101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.6263 - loss: 0.9440

I0000 00:00:1787654585.323416    1281 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 15222331332215370311
I0000 00:00:1787654585.323485    1281 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 182727892307534378


101/101 ━━━━━━━━━━━━━━━━━━━━ 11s 112ms/step - accuracy: 0.6263 - loss: 0.9440 - val_accuracy: 0.7114 - val_loss: 0.8772 - learning_rate: 9.0000e-05


I0000 00:00:1787654586.243918    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 5969345449245374614
I0000 00:00:1787654586.243971    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 18332283301707709952
I0000 00:00:1787654586.243977    1408 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 5255449077192355526


In [5]:
best_model = tf.keras.models.load_model(model_path)
actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
predicted = best_model.predict(test_dataset, verbose=0).argmax(axis=1)
metrics = {'model': 'custom_cnn', **calculate_classification_metrics(actual, predicted)}
print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0))
test.to_csv(ARTIFACT_DIR / 'models' / 'custom_cnn_test_manifest.csv', index=False)
(ARTIFACT_DIR / 'models' / 'custom_cnn_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', model_path)
print(metrics)

                    precision    recall  f1-score   support

       NO_FRACTURE       0.82      1.00      0.90       331
      ONE_FRACTURE       0.00      0.00      0.00        55
MULTIPLE_FRACTURES       0.00      0.00      0.00        17

          accuracy                           0.82       403
         macro avg       0.27      0.33      0.30       403
      weighted avg       0.67      0.82      0.74       403

Saved: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service/artifacts/models/custom_cnn.keras
{'model': 'custom_cnn', 'accuracy': 0.8213399503722084, 'macro_precision': 0.2737799834574028, 'macro_recall': 0.3333333333333333, 'macro_f1': 0.3006357856494096}
